# Jalon 3 & 4 — Baseline ML et Évaluation

## Objectif
Établir une baseline ML solide avant le Deep Learning, en utilisant des modèles linéaires régularisés :  
**Ridge (L2)**, **Lasso (L1)**, **ElasticNet (L1+L2)**.

Ces trois modèles permettent de démontrer la maîtrise du compromis **biais/variance** via le paramètre `alpha`.

**Variable cible** : `addiction_score` (régression).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from src.preprocessing import load_data, prepare_ml, FEATURES
from src.models_ml import train_ridge, train_lasso, train_elasticnet, plot_learning_curve, plot_regularization_path
from src.evaluation import regression_metrics, compare_models, plot_predictions, bias_variance_analysis

plt.rcParams['figure.dpi'] = 110

df = load_data('../data/tiktok_instagram_global_100countries.csv')
X_train, X_test, y_train, y_test, scaler, le = prepare_ml(df)

print(f'Train : {X_train.shape} | Test : {X_test.shape}')

## Rappel théorique — Régularisation

| Modèle | Pénalité | Effet |
|---|---|---|
| **Ridge** | $\lambda \sum w_i^2$ (L2) | Réduit tous les coefficients uniformément, ne les annule pas |
| **Lasso** | $\lambda \sum |w_i|$ (L1) | Sélection de features : force certains coefficients à 0 |
| **ElasticNet** | $\alpha (\rho \cdot L1 + (1-\rho) \cdot L2)$ | Combine les deux, utile quand il y a des features corrélées |

Un `alpha` élevé → régularisation forte → plus de biais, moins de variance → évite l'overfitting.

## Jalon 3 — Entraînement des 3 modèles
### 3.1 Ridge

In [ ]:
ridge_cv = train_ridge(X_train, y_train)
ridge_best = ridge_cv.best_estimator_

y_pred_ridge = ridge_best.predict(X_test)
metrics_ridge = regression_metrics(y_test, y_pred_ridge, 'Ridge')
print(metrics_ridge)

### 3.2 Lasso

In [ ]:
lasso_cv = train_lasso(X_train, y_train)
lasso_best = lasso_cv.best_estimator_

y_pred_lasso = lasso_best.predict(X_test)
metrics_lasso = regression_metrics(y_test, y_pred_lasso, 'Lasso')
print(metrics_lasso)

n_nonzero = (lasso_best.coef_ != 0).sum()
print(f'Features sélectionnées par Lasso : {n_nonzero}/{len(FEATURES)}')

### 3.3 ElasticNet

In [ ]:
en_cv = train_elasticnet(X_train, y_train)
en_best = en_cv.best_estimator_

y_pred_en = en_best.predict(X_test)
metrics_en = regression_metrics(y_test, y_pred_en, 'ElasticNet')
print(metrics_en)

### 3.4 Tableau comparatif

In [ ]:
results = [metrics_ridge, metrics_lasso, metrics_en]
print(compare_models(results))

### 3.5 Chemin de régularisation

In [ ]:
fig = plot_regularization_path(X_train, y_train, feature_names=FEATURES)
plt.show()
print('Observation : Lasso annule progressivement les features peu informatives (sélection sparse).')
print('Ridge les réduit sans jamais les mettre à 0 (préservation de toutes les features).')

## Jalon 4 — Évaluation et analyse Biais/Variance

### 4.1 Visualisation des prédictions

In [ ]:
best_model_name = compare_models(results).index[0]
preds_map = {'Ridge': y_pred_ridge, 'Lasso': y_pred_lasso, 'ElasticNet': y_pred_en}
fig = plot_predictions(y_test, preds_map[best_model_name], best_model_name)
plt.show()

### 4.2 Courbes d'apprentissage — Analyse biais/variance

In [ ]:
print('=== Ridge ===')
fig = plot_learning_curve(ridge_best, X_train, y_train, 'Courbe d\'apprentissage — Ridge')
plt.show()

print('=== Lasso ===')
fig = plot_learning_curve(lasso_best, X_train, y_train, 'Courbe d\'apprentissage — Lasso')
plt.show()

print('=== ElasticNet ===')
fig = plot_learning_curve(en_best, X_train, y_train, 'Courbe d\'apprentissage — ElasticNet')
plt.show()

### 4.3 Diagnostic biais/variance

In [ ]:
from sklearn.metrics import r2_score
for name, model, y_pred in [
    ('Ridge', ridge_best, y_pred_ridge),
    ('Lasso', lasso_best, y_pred_lasso),
    ('ElasticNet', en_best, y_pred_en),
]:
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2 = r2_score(y_test, y_pred)
    bias_variance_analysis(train_r2, test_r2, name)

### 4.4 Feature importance (coefficients Ridge)

In [ ]:
coef_df = pd.Series(ridge_best.coef_, index=FEATURES).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['salmon' if v > 0 else 'steelblue' for v in coef_df]
coef_df.plot.barh(ax=ax, color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coefficients Ridge (après scaling) — importance des features')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.show()

## Sauvegarde du meilleur modèle ML (baseline)

In [ ]:
models_map = {'Ridge': ridge_best, 'Lasso': lasso_best, 'ElasticNet': en_best}
best_ml_model = models_map[best_model_name]

joblib.dump(best_ml_model, f'../models/baseline_{best_model_name.lower()}.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(le, '../models/label_encoder.joblib')
print(f'Modèle {best_model_name} sauvegardé dans models/')

## Synthèse Baseline ML

Les modèles linéaires régularisés constituent une excellente baseline pour ce dataset synthétique tabulaire.  
Le meilleur modèle sera comparé avec le MLP (notebook 03) dans l'analyse ML vs DL (Jalon 7).  

**Limitation principale** : ces modèles ne capturent pas les non-linéarités ni les interactions temporelles entre années → motivation pour le Deep Learning.